# Object Detection using TensorFlow

**The split we're using in this project:**
- **Classification** (is it a cat or a dog?) - done by a **CNN we build and train ourselves**, from real cat/dog photos.
- **Localization** (where is the animal in the photo?) - done by a **pretrained detector** (SSD MobileNet V2, trained on COCO). This part is *not* trained by us, because real cat/dog datasets do not come with bounding box labels - nobody has hand-drawn a box around the animal in every training photo. Training our own box regressor would need a dataset that simply is not available here.

So the pretrained model never gets to decide "cat or dog" - it only draws a rough box around where an animal-shaped thing is. Our own trained CNN then looks specifically at what's inside that box and makes the actual cat/dog call.

1. Section 1 - Imports and setup
2. Section 2 - Visualization utilities
3. Section 3 - Data extraction and preprocessing (real cat/dog photos from `cats_vs_dogs`)
4. Section 4 - Building our own CNN
5. Section 5 - Training and evaluating our CNN
6. Section 6 - Loading a pretrained detector, used only to find the box
7. Section 7 - Combining both: detect the box, classify with our own CNN
8. Section 8 - The Dashboard, running and displayed right here at the end

## Quick Theory: Why Split Classification and Localization?

- **Classification** answers *"what is this?"* - it needs a labeled dataset where every image has a class (cat / dog). `cats_vs_dogs` gives us exactly that, so we can train our own model on it.
- **Localization** answers *"where is it in the image?"* - it needs a dataset where every image also has a hand-drawn box around the object. `cats_vs_dogs` does **not** include that, and drawing boxes on thousands of photos ourselves is not realistic for one class.
- So we use a **pretrained detector** (already trained on COCO, a dataset that does include boxes) purely as a "where roughly is an animal" finder, and let **our own trained CNN** make the actual cat/dog decision on just that region.
- This pattern - *reuse a pretrained model for the part you cannot easily train yourself, train your own model for the part you can* - is extremely common in real projects.

## Section 1 - Imports and Setup

### Why This Step Matters

We need three categories of tools here:
- **Modeling**: TensorFlow (to build/train our own CNN), TensorFlow Hub (to load the pretrained COCO detector), TensorFlow Datasets (to load `cats_vs_dogs` with zero manual downloading).
- **Image handling**: NumPy, OpenCV (drawing boxes/crops), Matplotlib (displaying things inline).
- **The dashboard**: Gradio - a library that turns a Python function into a shareable, interactive web UI with almost no code, and displays inline automatically in Colab via its built-in port-forwarding.

We also introduce a small but handy technique: checking an installed package's version **without importing the whole package**, using `importlib.metadata`.

### Variable & Function Dictionary

- `metadata.version(package_name)` *(function, from `importlib.metadata`)* — looks up a package's installed version directly from its installed distribution metadata, without triggering that package's actual Python import.
- `tf`, `hub`, `tfds`, `np`, `plt`, `cv2`, `gr` — TensorFlow, TensorFlow Hub, TensorFlow Datasets, NumPy, Matplotlib's plotting module, OpenCV, and Gradio respectively.

In [ ]:
# Install the libraries this notebook needs that aren't preinstalled in Colab
!pip install -q tensorflow_hub tensorflow_datasets opencv-python-headless gradio

# Import this helper so we can check a package's installed version without importing the package itself
from importlib import metadata

# Print a few key package versions using metadata.version(), before we even import them as modules
for package_name in ["tensorflow", "tensorflow-hub", "tensorflow-datasets", "opencv-python-headless", "gradio"]:
    try:
        print(f"{package_name}: {metadata.version(package_name)}")   # Look up and print the installed version
    except metadata.PackageNotFoundError:
        print(f"{package_name}: not installed")                       # Handle the case where it isn't installed yet

# Import TensorFlow, the core deep learning framework we'll use to build and train our own CNN
import tensorflow as tf

# Import TensorFlow Hub, used to load the pretrained SSD MobileNet V2 detector
import tensorflow_hub as hub

# Import TensorFlow Datasets, used to load the cats_vs_dogs dataset with no manual downloading
import tensorflow_datasets as tfds

# Import NumPy for numerical array operations
import numpy as np

# Import Matplotlib's plotting module for displaying images and training curves
import matplotlib.pyplot as plt

# Import OpenCV, used for cropping, drawing boxes, and resizing images
import cv2

# Import Gradio, used to build the final interactive dashboard
import gradio as gr

# Define AUTOTUNE: tells TensorFlow's data pipeline to automatically optimize parallel processing
AUTOTUNE = tf.data.AUTOTUNE

print("\nAll imports successful.")

## Section 2 - Visualization Utilities

### Why This Step Matters

Rather than rewriting plotting/drawing code in every later section, we define a small set of **reusable helper functions up front**. We'll need three kinds of visualization throughout this notebook:
1. Quickly displaying a single image with a title.
2. Drawing a labeled bounding box on an image (used in Sections 3, 7, and 8).
3. Plotting training/validation curves after training our CNN (used in Section 5).

### Variable & Function Dictionary

- `show_image(image, title=None, figsize=(6,6))` *(function)* — displays one image inline using Matplotlib.
- `draw_labeled_box(image, box, label=None, color=(0,255,0), thickness=2)` *(function)* — draws a bounding box (given in normalized `[ymin, xmin, ymax, xmax]` coordinates) and an optional text label directly onto an image array, returning the modified array.
- `plot_history(history, metric_name="accuracy")` *(function)* — plots training vs. validation curves for both loss and a given metric, side-by-side.

In [ ]:
# Define a small helper to quickly display a single image inline, with an optional title
def show_image(image, title=None, figsize=(6, 6)):
    plt.figure(figsize=figsize)   # Create a new figure of the requested size
    plt.imshow(image)              # Display the image array
    if title:                       # If a title was provided, show it above the image
        plt.title(title)
    plt.axis('off')                 # Hide axis ticks for a cleaner look
    plt.show()                      # Render the figure

# Define a helper to draw a labeled bounding box onto an image, given normalized [ymin, xmin, ymax, xmax]
def draw_labeled_box(image, box, label=None, color=(0, 255, 0), thickness=2):
    image = image.copy()                       # Copy the image so we never modify the caller's original array
    h, w, _ = image.shape                       # Get this image's pixel height and width
    ymin, xmin, ymax, xmax = box                 # Unpack the normalized coordinates
    left, right = int(xmin * w), int(xmax * w)    # Convert normalized x-coordinates to pixel columns
    top, bottom = int(ymin * h), int(ymax * h)     # Convert normalized y-coordinates to pixel rows
    cv2.rectangle(image, (left, top), (right, bottom), color, thickness)  # Draw the rectangle
    if label:                                       # If a label string was provided, draw it above the box
        cv2.putText(image, label, (left, max(top - 8, 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    return image                                     # Return the annotated copy

# Define a helper to plot training/validation loss and a chosen metric, side-by-side
def plot_history(history, metric_name="accuracy"):
    plt.figure(figsize=(12, 4))                        # Create a wide figure to hold two subplots

    plt.subplot(1, 2, 1)                                 # Left subplot: loss curves
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Loss')
    plt.xlabel('Epoch')
    plt.legend()

    plt.subplot(1, 2, 2)                                 # Right subplot: metric curves (e.g., accuracy)
    plt.plot(history.history[metric_name], label=f'Training {metric_name}')
    plt.plot(history.history[f'val_{metric_name}'], label=f'Validation {metric_name}')
    plt.title(metric_name.capitalize())
    plt.xlabel('Epoch')
    plt.legend()

    plt.show()

print("Visualization utilities defined: show_image, draw_labeled_box, plot_history")

## Section 3 - Data Extraction and Preprocessing

### Why This Step Matters

We now load the **real cat/dog photos** our CNN will actually learn from, directly from Hugging Face's **`microsoft/cats_vs_dogs`** dataset - the same underlying Microsoft Research Asirra photo set used by both the classic Kaggle competition and `tensorflow_datasets`' version, but loaded here through the `datasets` library, which needs **no API credentials and no manual download** at all (unlike Kaggle).

**Dataset fields (confirmed directly from the dataset's own loading script):**
- `image` - the photo, as a PIL Image (variable size, so we resize every one to a consistent `160x160`).
- `labels` - an integer class ID: `0 = cat`, `1 = dog`.

**A quirk of this dataset:** it ships with **only one split** (`train`) - no separate official validation or test split. So, just like our earlier attempts with `tensorflow_datasets` and Kaggle, we carve out our **own** 80% / 10% / 10% train/validation/test split - this time using the `datasets` library's built-in `.train_test_split()` method, applied twice (first 80/20, then splitting that 20% in half to get 10%/10%).

**Preprocessing steps:**
1. `resize_example(...)` converts every image to RGB (a few images in this dataset are grayscale/CMYK, which would otherwise break our CNN's expected 3-channel input) and resizes it to `160x160`, turning it into a plain NumPy array.
2. `.to_tf_dataset(...)` converts each Hugging Face split into a batched `tf.data.Dataset`, exactly like the pipelines used throughout the rest of this notebook.
3. We then rescale pixel values from `[0, 255]` to `[0, 1]` via `.map(normalize, ...)`, since we are **not** using any pretrained backbone here.

### Variable & Function Dictionary

- `IMG_SIZE` *(int)* — target height/width in pixels: `160`.
- `load_dataset("microsoft/cats_vs_dogs")` *(function, from `datasets`)* — downloads and loads the dataset, returning a `DatasetDict` (here containing only a `"train"` split).
- `full_dataset.train_test_split(test_size=..., seed=...)` *(method)* — splits a Hugging Face dataset into two parts; we call this twice to get our 80/10/10 split.
- `resize_example(example)` *(function)* — converts one example's `image` field to a fixed-size RGB NumPy array.
- `train_dataset` / `val_dataset` / `test_dataset` *(tf.data.Dataset objects)* — the raw (0-255 range) datasets, produced via `.to_tf_dataset(...)`.
- `normalize(image, label)` *(function)* — rescales one image's pixels from `[0,255]` to `[0,1]`.
- `train_ds` / `val_ds` / `test_ds` *(tf.data.Dataset objects)* — final, normalized, prefetched, model-ready pipelines.

In [ ]:
# Define the fixed image size (height and width) every image will be resized to
IMG_SIZE = 160

# Define the batch size for training
BATCH_SIZE = 32

# Install Hugging Face's datasets library, used to load microsoft/cats_vs_dogs with zero manual downloading
!pip install -q datasets

# Import the load_dataset function from Hugging Face's datasets library
from datasets import load_dataset

# Download and load the Microsoft Cats vs Dogs dataset; it ships with a single 'train' split only
raw_datasets = load_dataset("microsoft/cats_vs_dogs")
full_dataset = raw_datasets["train"]
print("Total examples:", len(full_dataset))

# Retrieve the human-readable class names directly from the dataset's own label metadata (['cat', 'dog'])
class_names = full_dataset.features["labels"].names
print("Classes:", class_names)

# Split the single 'train' split ourselves into 80% train / 10% val / 10% test
split_1 = full_dataset.train_test_split(test_size=0.2, seed=123)      # 80% train, 20% held out
train_hf = split_1["train"]
split_2 = split_1["test"].train_test_split(test_size=0.5, seed=123)   # Split that 20% into 10% / 10%
val_hf = split_2["train"]
test_hf = split_2["test"]

# Print how many examples ended up in each of our custom splits
print("Training examples:", len(train_hf))
print("Validation examples:", len(val_hf))
print("Test examples:", len(test_hf))

# Define a function that resizes one example's image to our fixed size and ensures 3 color channels
def resize_example(example):
    image = example["image"].convert("RGB").resize((IMG_SIZE, IMG_SIZE))  # Force RGB, resize to fixed size
    example["image"] = np.array(image)                                      # Convert the PIL image to a NumPy array
    return example

# Apply the resize function to every example in each split (the datasets library caches this after the first run)
train_hf = train_hf.map(resize_example)
val_hf = val_hf.map(resize_example)
test_hf = test_hf.map(resize_example)

# Set each dataset's output format to NumPy arrays, so they convert cleanly into TensorFlow tensors next
train_hf.set_format(type="numpy", columns=["image", "labels"])
val_hf.set_format(type="numpy", columns=["image", "labels"])
test_hf.set_format(type="numpy", columns=["image", "labels"])

# Convert each Hugging Face dataset into a batched tf.data.Dataset (raw, 0-255 pixel range)
train_dataset = train_hf.to_tf_dataset(columns="image", label_cols="labels", batch_size=BATCH_SIZE, shuffle=True)
val_dataset = val_hf.to_tf_dataset(columns="image", label_cols="labels", batch_size=BATCH_SIZE, shuffle=False)
test_dataset = test_hf.to_tf_dataset(columns="image", label_cols="labels", batch_size=BATCH_SIZE, shuffle=False)

# Define a function that rescales pixel values from [0,255] to [0,1]
def normalize(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

# Apply normalization and prefetching to build our final, model-ready training/validation/test pipelines
train_ds = train_dataset.map(normalize, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds = val_dataset.map(normalize, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds = test_dataset.map(normalize, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

# Visualize a sample batch, as a sanity check
for image_batch, label_batch in train_ds.take(1):
    plt.figure(figsize=(10, 10))
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(image_batch[i].numpy())              # Pixel values are already in [0,1], imshow handles this fine
        plt.title(class_names[int(label_batch[i].numpy())])   # Show the true class name
        plt.axis('off')
    plt.show()

## Section 4 - Building Our Own CNN

### Why This Step Matters

This is the heart of the "classification" half of our project: a **Convolutional Neural Network we build and train entirely from scratch** - no pretrained backbone this time.

**Architecture, block by block:**
- **Conv2D + MaxPooling2D, repeated 3 times, with increasing filter counts (32 -> 64 -> 128)**: each convolutional layer learns a set of small pattern-detecting filters; pooling then shrinks the spatial size.
- **Flatten**: converts the final 3D feature map into a single 1D vector.
- **Dense(128, relu) + Dropout(0.5)**: a fully-connected layer combining all extracted features, with heavy dropout to fight overfitting.
- **Dense(1, sigmoid)**: the final output neuron, producing a cat/dog probability.

### Variable & Function Dictionary

- `IMG_SHAPE` *(tuple)* — `(160, 160, 3)`.
- `layers.Conv2D(filters, kernel_size, activation)` *(layer)* — a convolutional layer; `filters` = number of pattern detectors, `kernel_size` = the detector's pixel window size.
- `layers.MaxPooling2D()` *(layer)* — shrinks the spatial dimensions by keeping only the strongest value in each small local region.
- `layers.Flatten()` *(layer)* — reshapes a 3D feature map into a 1D vector.
- `our_cnn` *(Keras Sequential model)* — our complete, from-scratch CNN.

In [ ]:
# Import the layers module for building our model architecture
from tensorflow.keras import layers, models

# Define the full input shape our CNN expects
IMG_SHAPE = (IMG_SIZE, IMG_SIZE, 3)

# Build our CNN as a Sequential stack of layers, entirely from scratch (no pretrained weights)
our_cnn = models.Sequential([
    layers.Input(shape=IMG_SHAPE),                                # Define the expected input shape

    layers.Conv2D(32, (3, 3), activation='relu'),                 # First conv block: 32 small pattern detectors
    layers.MaxPooling2D((2, 2)),                                    # Shrink spatial size, keep strongest signals

    layers.Conv2D(64, (3, 3), activation='relu'),                 # Second conv block: 64 more complex detectors
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu'),                # Third conv block: 128 even higher-level detectors
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),                                               # Flatten the 3D feature map into a 1D vector

    layers.Dense(128, activation='relu'),                          # Fully-connected layer combining all features
    layers.Dropout(0.5),                                            # Randomly disable 50% of neurons, to fight overfitting

    layers.Dense(1, activation='sigmoid')                          # Final output: probability of "dog" (0=cat, 1=dog)
])

# Print a full summary of the CNN's architecture and parameter counts
our_cnn.summary()

## Section 5 - Training and Evaluating Our CNN

### Why This Step Matters

We compile with the standard binary-classification setup (Adam optimizer, binary cross-entropy loss, accuracy metric), then train.

**A note on epoch count:** because this CNN starts from random weights (no pretrained knowledge), it needs meaningfully more epochs than a transfer-learning classifier to reach good accuracy. We'll run a modest **10 epochs** here to keep the live notebook run quick - feel free to bump this to 20-30+ afterward for a noticeably stronger model.

We then plot loss/accuracy curves using our Section 2 utility, and run one final, objective evaluation on `test_ds`.

### Variable & Function Dictionary

- `EPOCHS` *(int)* — number of full training passes (`10`).
- `history` *(Keras History object)* — records loss/accuracy per epoch for both training and validation.

In [ ]:
# Compile our CNN: Adam optimizer, binary cross-entropy loss (matches our sigmoid output), track accuracy
our_cnn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Define how many full passes through the training dataset we'll perform
EPOCHS = 10

# Train the CNN from scratch on our real cat/dog photos
history = our_cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

# Plot the training/validation loss and accuracy curves using our Section 2 utility
plot_history(history, metric_name="accuracy")

# Run a final, unbiased evaluation on the held-out test set
test_loss, test_accuracy = our_cnn.evaluate(test_ds)
print(f"\nFinal test loss: {test_loss:.4f}")
print(f"Final test accuracy: {test_accuracy:.4f}")

## Section 6 - Loading a Pretrained Detector, Used Only to Find the Box

### Why This Step Matters

Now we bring in **SSD MobileNet V2**, pretrained on COCO. But this time, we use it very differently: **we deliberately ignore its predicted class entirely.** We only care about *one* thing from its output: *"where is the single most confident detection in this image?"* - whatever it thinks that object is.

**Why ignore its class predictions?**
Because this detector was never trained specifically to be an expert on cats vs. dogs - it just happens to recognize "cat" and "dog" as 2 of its 80 COCO categories, among many others. We don't need it to be right about *what* the animal is - we only need it to draw a reasonably good box around *where* the animal is. The actual cat/dog decision is reserved entirely for our own trained CNN in Section 7.

**The fallback case:** if the detector finds nothing confidently enough (below our `confidence_threshold`), we simply fall back to treating the **entire image** as "the box".

### Variable & Function Dictionary

- `detector` *(callable model object)* — the loaded SSD MobileNet V2 network from TensorFlow Hub.
- `locate_animal(image_np, confidence_threshold=0.3)` *(function)* — runs the detector and returns the single highest-confidence bounding box (normalized `[ymin, xmin, ymax, xmax]`), or `[0, 0, 1, 1]` (the whole image) if nothing meets the threshold.

In [ ]:
# %%time measures and prints how long this cell takes to execute
%%time

# Define the URL of the pretrained SSD MobileNet V2 model on TensorFlow Hub (trained on COCO, 80 classes)
model_url = "https://tfhub.dev/tensorflow/ssd_mobilenet_v2/2"

# Load the model into memory; 'detector' behaves like a callable function representing the full network
print("Loading pretrained detector... please wait.")
detector = hub.load(model_url)
print("Detector loaded successfully!")

# Define a function that finds the single most confident detection in an image, regardless of predicted class
def locate_animal(image_np, confidence_threshold=0.3):
    # Add a batch dimension and convert to a uint8 tensor, exactly as the detector expects
    input_tensor = tf.convert_to_tensor(np.expand_dims(image_np, axis=0), dtype=tf.uint8)

    # Run the detector; this returns a dictionary of raw output tensors
    result = detector(input_tensor)

    # Convert the output tensors to NumPy arrays for easy indexing
    scores = result['detection_scores'][0].numpy()   # Confidence scores for every candidate box
    boxes = result['detection_boxes'][0].numpy()      # Normalized [ymin, xmin, ymax, xmax] for every candidate box

    # Find the index of the single highest-confidence detection
    best_index = np.argmax(scores)
    best_score = scores[best_index]

    # If even the best detection is below our confidence threshold, fall back to using the whole image
    if best_score < confidence_threshold:
        return np.array([0.0, 0.0, 1.0, 1.0]), best_score  # [ymin, xmin, ymax, xmax] covering the entire image

    # Otherwise, return that best-scoring box and its confidence score
    return boxes[best_index], best_score

print("locate_animal() is ready to use.")

## Section 7 - Combining Both: Detect the Box, Classify with Our Own CNN

### Why This Step Matters

This is where the two halves of the project finally meet. The full pipeline, per image:
1. **Locate** (Section 6's pretrained detector) -> get a rough bounding box around the animal.
2. **Crop** that region out of the original image (with a small padding margin).
3. **Classify** (Section 4/5's own CNN) -> resize the crop to `160x160`, rescale to `[0,1]`, and predict cat vs. dog - *only* looking at the cropped region.
4. **Draw** the box and the predicted label back onto the original image.

This mirrors a real, common pattern in production computer vision systems: a fast, general-purpose **detector** narrows down *where* to look, and a smaller, specialized **classifier** makes the fine-grained decision on just that region.

### Variable & Function Dictionary

- `crop_with_padding(image_np, box, padding_frac=0.1)` *(function)* — crops the region defined by `box`, expanded outward by 10% of the box's own height/width in each direction (clipped to stay inside the image).
- `detect_and_classify(image_np)` *(function)* — runs the full pipeline described above; returns the annotated image, the predicted label, and the classifier's confidence.

In [ ]:
# Define a function that crops a region from an image, with a small padding margin around the box
def crop_with_padding(image_np, box, padding_frac=0.1):
    h, w, _ = image_np.shape                        # Get the image's pixel height/width
    ymin, xmin, ymax, xmax = box                      # Unpack the normalized box coordinates
    box_h, box_w = ymax - ymin, xmax - xmin             # Compute the box's own height/width, as fractions

    # Expand the box outward by padding_frac of its own size, clipped to stay within [0, 1]
    ymin_p = max(0.0, ymin - padding_frac * box_h)
    ymax_p = min(1.0, ymax + padding_frac * box_h)
    xmin_p = max(0.0, xmin - padding_frac * box_w)
    xmax_p = min(1.0, xmax + padding_frac * box_w)

    # Convert the padded, normalized coordinates into pixel indices
    top, bottom = int(ymin_p * h), int(ymax_p * h)
    left, right = int(xmin_p * w), int(xmax_p * w)

    # Return the cropped region of the image
    return image_np[top:bottom, left:right]

# Define the full combined pipeline: locate, crop, classify, and annotate
def detect_and_classify(image_np):
    # Step 1: Locate the animal using the pretrained detector (Section 6)
    box, detector_confidence = locate_animal(image_np)

    # Step 2: Crop that region out of the original image, with a small padding margin
    cropped = crop_with_padding(image_np, box)

    # Step 3: Resize the crop to our CNN's expected input size and rescale to [0, 1]
    cropped_resized = tf.image.resize(cropped, (IMG_SIZE, IMG_SIZE))
    cropped_normalized = tf.cast(cropped_resized, tf.float32) / 255.0

    # Add a batch dimension, since our CNN expects a batch of images
    cropped_batch = tf.expand_dims(cropped_normalized, axis=0)

    # Step 4: Run our own trained CNN on just the cropped region to get the cat/dog probability
    prediction = our_cnn.predict(cropped_batch, verbose=0)[0][0]
    predicted_class = class_names[int(round(prediction))]   # Threshold at 0.5 to get a class name
    confidence = prediction if predicted_class == 'dog' else 1 - prediction  # Confidence in the predicted class

    # Step 5: Draw the box and the predicted label back onto the original image
    label_text = f"{predicted_class}: {confidence:.2f}"
    annotated_image = draw_labeled_box(image_np, box, label=label_text, color=(0, 255, 0))

    return annotated_image, predicted_class, confidence

# Test the combined pipeline on a few real photos taken directly from our held-out test set.
# NOTE: we deliberately use 'test_dataset' here (the RAW, 0-255 range version, before normalization),
# since detect_and_classify()/locate_animal() expect uint8 pixel values, not the [0,1] floats in 'test_ds'.
plt.figure(figsize=(14, 10))
for image_batch, label_batch in test_dataset.take(1):     # Grab a single batch of un-normalized test images
    for i in range(6):                                       # Look at the first 6 images in that batch
        image_np = image_batch[i].numpy().astype("uint8")       # Convert to a uint8 NumPy array (0-255 range)
        annotated, pred_label, conf = detect_and_classify(image_np)  # Run the full pipeline

        true_name = class_names[int(label_batch[i].numpy())]      # Look up the true class name

        plt.subplot(2, 3, i + 1)
        plt.imshow(annotated)
        color = "green" if pred_label == true_name else "red"
        plt.title(f"Pred: {pred_label} ({conf:.2f})\nTrue: {true_name}", color=color)
        plt.axis('off')
plt.show()

## Section 8 - The Dashboard

### Why This Step Matters

The final piece: a genuinely **interactive dashboard**, so anyone can upload their own photo and immediately see the combined pipeline in action.

**How Gradio works here:** `gr.Interface` wraps a plain Python function into a web UI automatically. Calling `.launch()` starts a small local web server; in **Google Colab**, this is automatically displayed inline via Colab's built-in port-forwarding - in Jupyter/VS Code, it displays the same way or opens in your browser.

### Variable & Function Dictionary

- `dashboard_fn(image)` *(function)* — a thin wrapper around `detect_and_classify` matching Gradio's expected function signature: takes one image in, returns `(annotated_image, label_text)`.
- `demo` *(Gradio Interface object)* — the assembled dashboard app.
- `demo.launch()` *(method)* — starts the local server and displays the dashboard inline.

In [ ]:
# Define a thin wrapper function matching Gradio's expected signature: one image in, matching outputs out
def dashboard_fn(image):
    annotated_image, predicted_class, confidence = detect_and_classify(image)   # Run our full pipeline
    label_text = f"Prediction: {predicted_class} (confidence: {confidence:.2f})"  # Build a readable result string
    return annotated_image, label_text                                            # Return both outputs

# Build the Gradio dashboard: one image input, one annotated image output, one text output
demo = gr.Interface(
    fn=dashboard_fn,                                    # The function to run on every submitted photo
    inputs=gr.Image(type="numpy", label="Upload a photo of a cat or dog"),  # Accepts an uploaded image as a NumPy array
    outputs=[
        gr.Image(type="numpy", label="Detected & Classified"),  # Shows the annotated image with box + label
        gr.Textbox(label="Result")                                # Shows the prediction as readable text
    ],
    title="Cat vs. Dog: Detect the Box, Classify with Our Own CNN",
    description=(
        "Upload any photo. A pretrained SSD MobileNet V2 finds the animal's location; "
        "our own from-scratch-trained CNN then looks only at that cropped region to decide cat or dog."
    )
)

# Launch the dashboard; in Colab this displays inline automatically via port-forwarding
demo.launch()

## Wrap Up

**What we built, section by section:**
1-2. Set up our tools and reusable visualization helpers.
3. Loaded real cat/dog photos (`cats_vs_dogs` - the same source as Kaggle's "Dogs vs Cats") and built a proper 80/10/10 data pipeline.
4-5. Built and trained a CNN **entirely from scratch** to answer "cat or dog?".
6. Loaded a pretrained COCO detector, used **only** to answer "where's the animal?" - its own class predictions were deliberately ignored.
7. Combined both: detector finds the box -> we crop it -> our CNN classifies just that region.
8. Wrapped the whole pipeline in a live, interactive Gradio dashboard.

**The core lesson:** when a task needs two different kinds of labels (a class label *and* a box label) and you only have one of them for your specific objects, you don't need to give up on localization - you can borrow a general-purpose pretrained detector for the part you can't label yourself, and train a small specialized model for the part you can. This division of labor is genuinely how many real-world computer vision systems are built.